In [1]:
import pandas as pd

ImportError: C extension: pandas.compat not built. If you want to import pandas from the source directory, you may need to run 'python -m pip install -ve . --no-build-isolation -Ceditable-verbose=true' to build the C extensions first.

In [ ]:
df = pd.read_csv('Dataset\student_habits_performance.csv')

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
sns.set(style='whitegrid')

In [ ]:
df.isna().sum().sum()

In [ ]:
df.dropna(inplace=True)

In [ ]:
df.isna().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df.describe()

In [ ]:
df.describe(include="object").columns

In [ ]:
categorical_cols = ["gender", "part_time_job", "diet_quality", "parental_education_level", "internet_quality", "extracurricular_participation"]

In [ ]:
for col in categorical_cols:
    print(f"Value count for {col} : \n {df[col].value_counts()}")

In [ ]:
df.hist(bins=20, edgecolor="black")
plt.tight_layout()
plt.show()

In [ ]:
for col in categorical_cols:
    sns.countplot(data=df, x=col)
    plt.title(f"Distribution of {col}")
    plt.xticks(rotation=45)
    plt.show()

In [ ]:
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Matrix")
plt.show()

In [ ]:
df.describe().columns

In [ ]:
num_features = ['age', 'study_hours_per_day', 'social_media_hours', 'netflix_hours',
       'attendance_percentage', 'sleep_hours', 'exercise_frequency',
       'mental_health_rating']

In [ ]:
for features in num_features:
    sns.scatterplot(data=df, x=features, y="exam_score")
    plt.title(f"{features} vs Exam Score")
    plt.show()

In [ ]:
for col in categorical_cols:
    sns.boxplot(data=df, x=col, y="exam_score")
    plt.title(f"Exam Score by {col}")
    plt.xticks(rotation=45)
    plt.show()


In [ ]:
##Feature Engineering & Training ML Model
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor

In [ ]:
df.columns

In [ ]:
df.head(2)

In [ ]:
features = ["study_hours_per_day", "attendance_percentage", "mental_health_rating", "sleep_hours", "part_time_job"]

In [ ]:
target = "exam_score"

In [ ]:
df_model = df[features + [target]].copy()

In [ ]:
df_model

In [ ]:
le = LabelEncoder()

In [ ]:
df_model["part_time_job"] = le.fit_transform(df_model["part_time_job"])

In [ ]:
df_model

In [ ]:
X = df_model[features]

In [ ]:
y = df_model[target]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [ ]:
len(y_test)

In [ ]:
len(y_train)

In [ ]:
models = {
    "LinearRegression" : {
        "model" : LinearRegression(),
        "params" : {}
    },
    "Decision Tree": {
        "model": DecisionTreeRegressor(),
        "params": {
            "max_depth": [3,5,7,10],
            "min_samples_split": [2,4,6,8],
            "min_samples_leaf": [1,2,3]
        }
    },
    "RandomForest" : {
        "model" : RandomForestRegressor(),
        "params" : {"n_estimators" : [50, 100], "max_depth" : [5, 10]}
    }

}

In [ ]:
best_models = []

In [ ]:
for name, config in models.items():
    print(f"Training {name}")
    grid = GridSearchCV(config["model"],config["params"], cv=5, scoring="neg_mean_squared_error")
    grid.fit(X_train, y_train)

    y_pred = grid.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    best_models.append({
        "model" : name,
        "best_params" : grid.best_params_,
        "rmse" : rmse,
        "r2" : r2
    })


In [ ]:
results_df = pd.DataFrame(best_models)

In [ ]:
results_df.sort_values(by="rmse")

In [ ]:
import joblib

best_row = results_df.sort_values(by="rmse").iloc[0]


In [ ]:
best_row

In [ ]:
best_model_name = best_row["model"]

In [ ]:
best_model_name

In [ ]:
best_model_config = models[best_model_name]

In [ ]:
best_model_config

In [ ]:
final_model = best_model_config["model"]

In [ ]:
final_model.fit(X, y)

In [ ]:
final_model.predict(X_test)

In [ ]:
joblib.dump(final_model, "best_model.pkl")

In [ ]:
joblib.load("best_model.pkl").predict(X_test)